
# BM Supermercados and Consum API Client

This notebook collects product and price information from the BM and Consum JSON APIs.

## Collection process

1. Searches for the product terms defined in `SEARCH_TERMS`.
2. Retrieves all available pages with retries.
3. Extracts product IDs, EANs, brands, prices, offers, quantities, and timestamps.
4. Preserves the original API price fields in RAW format.
5. Validates the completeness of both supermarket collections.
6. Saves current RAW files and dated historical RAW snapshots.

## Outputs

### BM
- Current RAW: `data/raw/bm_products_raw.csv`
- Historical RAW: `data/history/bm/YYYY-MM-DD_raw.csv`

### Consum
- Current RAW: `data/raw/consum_products_raw.csv`
- Historical RAW: `data/history/consum/YYYY-MM-DD_raw.csv`

The central pipeline handles price normalization, promotion validation, deduplication, classification, and historical analysis.

## Collection policy

- Both supermarket collections must complete successfully before saving.
- Existing historical snapshots are never overwritten.
- TEST_MODE allows testing selected searches without saving files.


In [ ]:
import re
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests

CHAINS = {
    "BM": "https://www.online.bmsupermercados.es",
    "Consum": "https://tienda.consum.es",
}
SEARCH_PATH = "/api/rest/V1.0/catalog/product"

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------
TEST_MODE = False
TEST_TERMS = ["leche"]

TERMS_TO_FETCH = TEST_TERMS if TEST_MODE else SEARCH_TERMS

PAGE_SIZE = 100          # maximum the API accepts
REQUEST_PAUSE = 1.0      # seconds between requests
MAX_RETRIES = 3
TIMEOUT = 20
API_RESULT_CAP = 500     # the API never returns more than 500 results per search

HEADERS = {
    "User-Agent": "UC3M-WebAnalytics-StudentProject/1.0 (academic, non-commercial)",
    "Accept": "application/json",
}

print("Searches to run:", TERMS_TO_FETCH)

Searches to run: ['leche']


In [2]:
# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()

    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path

    raise FileNotFoundError("Could not locate the project root.")


BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"
HISTORY_DIR = BASE_DIR / "data" / "history"

RUN_DATE = datetime.now().strftime("%Y-%m-%d")

RAW_PATHS = {
    supermarket: RAW_DIR / f"{supermarket.lower()}_products_raw.csv"
    for supermarket in CHAINS
}

RAW_HISTORY_PATHS = {
    supermarket: (
        HISTORY_DIR / supermarket.lower() / f"{RUN_DATE}_raw.csv"
    )
    for supermarket in CHAINS
}

print("Project:", BASE_DIR)
print("Collection date:", RUN_DATE)

for supermarket in CHAINS:
    print(f"\n{supermarket}")
    print("Current RAW:", RAW_PATHS[supermarket])
    print("Historical RAW:", RAW_HISTORY_PATHS[supermarket])

Project: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Collection date: 2026-11-13

BM
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\bm_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\bm\2026-11-13_raw.csv

Consum
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\consum_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\consum\2026-11-13_raw.csv


## API requests

`get_page` makes one request with retries (exponential backoff on network errors,
429 or 5xx). `search_all` walks through every page of a term until `hasMore` is
false.

In [3]:
session = requests.Session()
session.headers.update(HEADERS)


def get_page(base_url, term, offset):
    url = base_url + SEARCH_PATH
    params = {"q": term, "limit": PAGE_SIZE, "offset": offset}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, params=params, timeout=TIMEOUT)

            if response.status_code == 429 or response.status_code >= 500:
                raise requests.HTTPError(f"HTTP {response.status_code}")

            response.raise_for_status()
            return response.json()

        except (requests.RequestException, ValueError) as exc:
            if attempt == MAX_RETRIES:
                raise
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({exc}). Retrying in {wait} s...")
            time.sleep(wait)


def search_all(base_url, term):
    products = []
    offset = 0
    total = None

    while True:
        data = get_page(base_url, term, offset)
        total = data.get("totalCount", 0)
        page = data.get("products", [])
        products.extend(page)

        time.sleep(REQUEST_PAUSE)

        if not data.get("hasMore") or not page:
            break
        offset += PAGE_SIZE

    return products, total

## Flatten the JSON

Each product is a nested JSON object. We keep the useful fields:

- `priceData.prices`: a list with `PRICE` (regular price) and, when there is a
  promotion, `OFFER_PRICE`. Despite its name, `centAmount` is already **in euros**.
- `centUnitAmount`: price per reference unit, described in `unitPriceUnitType`
  (BM uses `LITRO`/`KILO`; Consum uses `1 L`, `1 Kg`...).

In [4]:
def get_price(prices, price_id):
    for price in prices:
        if price.get("id") == price_id:
            value = price.get("value", {})
            return value.get("centAmount"), value.get("centUnitAmount")
    return None, None


def flatten_product(product, supermarket, searched_item, scraped_at):
    data = product.get("productData", {})
    price_data = product.get("priceData", {})
    prices = price_data.get("prices", [])

    regular_price, regular_unit_price = get_price(prices, "PRICE")
    offer_price, offer_unit_price = get_price(prices, "OFFER_PRICE")

    categories = product.get("categories") or []

    return {
        "searched_item": searched_item,
        "supermarket": supermarket,
        "product_id": product.get("code"),
        "ean": product.get("ean"),
        "brand": (data.get("brand") or {}).get("name"),
        "name": data.get("name"),
        "description": data.get("description"),
        "category": categories[0].get("name") if categories else None,
        "regular_price": regular_price,
        "offer_price": offer_price,
        "regular_unit_price": regular_unit_price,
        "offer_unit_price": offer_unit_price,
        "price_unit_type": price_data.get("priceUnitType"),
        "unit_price_unit_type": price_data.get("unitPriceUnitType"),
        "available": data.get("availability") == 1
                     and not data.get("temporaryOutOfStock", False),
        "url": data.get("url"),
        "image_url": data.get("imageURL"),
        "scraped_at": scraped_at,
    }

## Single-search test

Before running every search, we check with `leche` on both chains that the number of
rows matches the `totalCount` returned by the API.

In [5]:
for supermarket, base_url in CHAINS.items():
    products, total = search_all(base_url, "leche")
    scraped_at = datetime.now().isoformat(timespec="seconds")
    df_test = pd.DataFrame(
        [flatten_product(p, supermarket, "leche", scraped_at) for p in products]
    )
    print(f"{supermarket}: totalCount={total}, rows={len(df_test)}")
    display(df_test[["name", "brand", "regular_price", "offer_price",
                     "regular_unit_price", "unit_price_unit_type"]].head(5))

BM: totalCount=306, rows=306


,name,brand,regular_price,offer_price,regular_unit_price,unit_price_unit_type
0,Leche entera 1 l,BM,0.96,NaN,0.96,LITRO
1,"Leche entera 1,5 l",Pascual,1.98,NaN,1.32,LITRO
2,Leche entera 1 l,Kaiku,1.25,NaN,1.25,LITRO
3,"Leche entera 1,5 l",BM,1.56,NaN,1.04,LITRO
4,"Leche entera 1,5 l",Kaiku,1.89,NaN,1.26,LITRO


Consum: totalCount=303, rows=303


,name,brand,regular_price,offer_price,regular_unit_price,unit_price_unit_type
0,Leche Entera Brik,CONSUM,0.96,NaN,0.96,1 L
1,Leche Semidesnatada Brik,CONSUM,0.83,NaN,0.83,1 L
2,Leche Calcio Semidesnatada Brik,CONSUM,1.05,NaN,1.05,1 L
3,Leche Calcio Desnatada Brik,PULEVA,1.59,NaN,1.59,1 L
4,Leche sin Lactosa Entera Brik,CONSUM,1.03,NaN,1.03,1 L


## All searches

If a search still fails after the retries, it is logged as `ERROR` in the summary
and the loop moves on to the next one.

In [6]:
all_results = {supermarket: [] for supermarket in CHAINS}
summary = []

for supermarket, base_url in CHAINS.items():
    for i, term in enumerate(TERMS_TO_FETCH, start=1):
        print(f"[{supermarket}] [{i}/{len(TERMS_TO_FETCH)}] {term}")
        started = time.monotonic()

        try:
            products, total = search_all(base_url, term)
            scraped_at = datetime.now().isoformat(timespec="seconds")
            rows = [
                flatten_product(p, supermarket, term, scraped_at)
                for p in products
            ]
            all_results[supermarket].append(pd.DataFrame(rows))
            if len(rows) != total:
                status = "INCOMPLETE"
            elif total >= API_RESULT_CAP:
                status = "OK (500 cap)"
            else:
                status = "OK"

        except Exception as exc:
            print(f"  ERROR: {type(exc).__name__}: {exc}")
            rows, total, status = [], None, "ERROR"

        summary.append({
            "supermarket": supermarket,
            "searched_item": term,
            "total_count": total,
            "rows_extracted": len(rows),
            "status": status,
            "seconds": round(time.monotonic() - started, 2),
        })

df_summary = pd.DataFrame(summary)
display(df_summary)
print(df_summary.groupby(["supermarket", "status"]).size())

[BM] [1/1] leche
[Consum] [1/1] leche


,supermarket,searched_item,total_count,rows_extracted,status,seconds
0,BM,leche,306,306,OK,5.98
1,Consum,leche,303,303,OK,4.94


supermarket  status
BM           OK        1
Consum       OK        1
dtype: int64


In [7]:

# --------------------------------------------------
# PREPARE RAW DATA (one dataframe per chain)
# --------------------------------------------------

raw = {}

for supermarket, frames in all_results.items():
    if not frames:
        print(f"{supermarket}: no data collected.")
        continue

    raw[supermarket] = pd.concat(
        frames, ignore_index=True
    )

    print(
        f"{supermarket}: "
        f"{len(raw[supermarket])} raw rows ready."
    )


BM: 306 raw rows ready.
Consum: 303 raw rows ready.


In [8]:
# --------------------------------------------------
# VALIDATE AND SAVE BM AND CONSUM RAW DATA
# --------------------------------------------------

import os
import tempfile

if TEST_MODE:
    print("TEST MODE: no files will be saved.")

else:
    # --------------------------------------------------
    # VALIDATE COLLECTION COMPLETENESS
    # --------------------------------------------------

    if set(raw) != set(CHAINS):
        raise RuntimeError(
            "Missing supermarket data. No files saved."
        )

    expected_searches = len(CHAINS) * len(SEARCH_TERMS)

    if len(df_summary) != expected_searches:
        raise RuntimeError(
            "Not all expected searches were attempted. "
            "No files saved."
        )

    successful_statuses = ["OK", "OK (500 cap)"]

    if not df_summary["status"].isin(successful_statuses).all():
        raise RuntimeError(
            "Incomplete or failed searches. No files saved."
        )

    for supermarket in CHAINS:
        terms = df_summary.loc[
            df_summary["supermarket"] == supermarket,
            "searched_item"
        ]

        if (
            len(terms) != len(SEARCH_TERMS)
            or set(terms) != set(SEARCH_TERMS)
        ):
            raise RuntimeError(
                f"{supermarket}: missing or repeated searches. "
                "No files saved."
            )

    # --------------------------------------------------
    # VALIDATE RAW DATA
    # --------------------------------------------------

    required_columns = [
        "searched_item",
        "supermarket",
        "product_id",
        "ean",
        "brand",
        "name",
        "regular_price",
        "offer_price",
        "regular_unit_price",
        "offer_unit_price",
        "unit_price_unit_type",
        "scraped_at",
    ]

    for supermarket in CHAINS:
        df_raw = raw[supermarket]

        if df_raw.empty:
            raise RuntimeError(
                f"{supermarket}: empty RAW dataset. "
                "No files saved."
            )

        missing_columns = [
            col for col in required_columns
            if col not in df_raw.columns
        ]

        if missing_columns:
            raise RuntimeError(
                f"{supermarket}: missing RAW columns "
                f"{missing_columns}"
            )

        regular_prices = pd.to_numeric(
            df_raw["regular_price"],
            errors="coerce"
        )

        offer_prices = pd.to_numeric(
            df_raw["offer_price"],
            errors="coerce"
        )

        # Use a positive offer price when available.
        effective_prices = offer_prices.where(
            offer_prices.gt(0),
            regular_prices
        )

        valid_names = (
            df_raw["name"].notna()
            & df_raw["name"].astype(str).str.strip().ne("")
        )

        valid_prices = (
            effective_prices.notna()
            & effective_prices.gt(0)
        )

        if not (valid_names & valid_prices).any():
            raise RuntimeError(
                f"{supermarket}: no valid products found. "
                "No files saved."
            )

        invalid_rows = int(
            (~(valid_names & valid_prices)).sum()
        )

        print(
            f"{supermarket}: {len(df_raw)} RAW rows, "
            f"{invalid_rows} rows requiring validation."
        )

    # --------------------------------------------------
    # PROTECT EXISTING HISTORICAL SNAPSHOTS
    # --------------------------------------------------

    for supermarket in CHAINS:
        path = RAW_HISTORY_PATHS[supermarket]

        if path.exists():
            raise FileExistsError(
                f"{supermarket}: historical RAW already exists "
                f"for {RUN_DATE}: {path}. "
                "No files were modified."
            )

    # --------------------------------------------------
    # PREPARE TEMPORARY FILES
    # --------------------------------------------------

    def prepare_raw_temp(frame, destination):
        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        fd, temp_name = tempfile.mkstemp(
            prefix=".api_raw_",
            suffix=".tmp",
            dir=destination.parent
        )

        try:
            with os.fdopen(
                fd,
                "w",
                encoding="utf-8-sig",
                newline=""
            ) as file:
                frame.to_csv(file, index=False)

            return Path(temp_name)

        except Exception:
            Path(temp_name).unlink(missing_ok=True)
            raise

    history_temps = {}
    current_temps = {}
    created_histories = []

    try:
        for supermarket in CHAINS:
            history_temps[supermarket] = prepare_raw_temp(
                raw[supermarket],
                RAW_HISTORY_PATHS[supermarket]
            )

            current_temps[supermarket] = prepare_raw_temp(
                raw[supermarket],
                RAW_PATHS[supermarket]
            )

        # Create both historical files without overwriting.
        for supermarket in CHAINS:
            history_path = RAW_HISTORY_PATHS[supermarket]

            with history_temps[supermarket].open("rb") as source:
                with history_path.open("xb") as target:
                    created_histories.append(history_path)

                    while True:
                        chunk = source.read(1024 * 1024)

                        if not chunk:
                            break

                        target.write(chunk)

            print(
                f"{supermarket}: historical RAW saved:",
                history_path
            )

        # Update current RAW files only after both histories exist.
        for supermarket in CHAINS:
            os.replace(
                current_temps[supermarket],
                RAW_PATHS[supermarket]
            )

            print(
                f"{supermarket}: current RAW updated:",
                RAW_PATHS[supermarket]
            )

    except Exception:
        print(
            "ERROR: RAW saving was interrupted. "
            "Review current and historical files before retrying."
        )
        raise

    finally:
        for temp_path in [
            *history_temps.values(),
            *current_temps.values()
        ]:
            temp_path.unlink(missing_ok=True)

    # --------------------------------------------------
    # COLLECTION SUMMARY
    # --------------------------------------------------

    print("\nCOLLECTION COMPLETED")
    print("Date:", RUN_DATE)
    print("Status: COMPLETE")

    for supermarket in CHAINS:
        df_raw = raw[supermarket]

        print(f"\n{supermarket}")
        print("RAW rows:", len(df_raw))
        print(
            "Distinct product IDs:",
            df_raw["product_id"].nunique()
        )
        print("Current RAW:", RAW_PATHS[supermarket])
        print("Historical RAW:", RAW_HISTORY_PATHS[supermarket])

TEST MODE: no files will be saved.
